# unbox-args-tensor-to-array — ex1: unbox MiniTensor positional args to raw arrays, pass-through non-Tensors

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbox-args-tensor-to-array`. Running the final beacon cell reports progress against the `Backprop: Unbox Tensor args to array` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbox Tensor args to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbox-args-tensor-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbox-args-tensor-to-array"
DD_SUBTOPIC = "Backprop: Unbox Tensor args to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Unbox Tensor args → raw arrays — quick refresher

The **first half** of `wrap_forward_fn` strips wrappers off positional args so the underlying `fwd_fn` (e.g. `torch.log`) sees raw `torch.Tensor` (or scalars / shape tuples) — it has no idea our `Tensor` class exists:

```python
raw_args = tuple(
    a.array if isinstance(a, Tensor) else a
    for a in args
)
out_raw = fwd_fn(*raw_args, **kwargs)
```

Two rules:
- **`isinstance(a, Tensor)` is the gate.** Anything else (int, float,   tuple, ndarray) passes through untouched.
- **Read `.array`, never copy.** The raw tensor stays the same object   — the Recipe later stores these same raw tensors for replay;   cloning would burn memory and break identity invariants.

Dual of `build_parents`: where `build_parents` *keeps* the wrappers (filtered, keyed by argnum), `unbox_args` *replaces* them with their `.array`. Same `isinstance` check, opposite transform.

### Exercise 1 — unbox MiniTensor positional args to raw arrays, pass-through non-Tensors

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the unboxing half of wrap_forward_fn: replace each MiniTensor arg with its `.array`, leave non-Tensors untouched, preserve order.
> Keywords: unbox, wrap-forward, isinstance, raw-array
> ```

**KCs targeted:** `unbox-args-tensor-to-array`, `parents-dict-by-argidx`

Implement `unbox_args(args)`. Given a tuple of positional inputs (some `MiniTensor`, some Python scalars / ndarray / shape tuples), return a NEW tuple where every `MiniTensor` has been replaced by its `.array`, in the same positions:

```
unbox_args((t1, 3.0, t2))   == (t1.array, 3.0, t2.array)
unbox_args((5, t1, 'x'))    == (5, t1.array, 'x')
unbox_args(())              == ()
```

This is the FIRST half of `wrap_forward_fn`. The underlying raw fn (e.g. `torch.log`) doesn't know our `MiniTensor` class exists — it expects plain `torch.Tensor`. The unbox step bridges the wrapper-world to the raw-world before the call.

Two rules:

**1. `isinstance(a, MiniTensor)` is the gate.** Anything else passes through unchanged. Don't duck-type on `.array` — random objects can have an `.array` attribute (numpy ndarrays do, via the array protocol).

**2. Read `.array` directly — don't copy.** The raw tensor IS the same object. The Recipe will store it; copying would burn memory and break identity invariants used by `is`-checks later.

Canonical one-liner: `tuple(a.array if isinstance(a, MiniTensor) else a for a in args)`.

In [ ]:
def unbox_args(args: tuple) -> tuple:
    """Replace each MiniTensor in args with its .array; preserve order + non-Tensors."""
    raise NotImplementedError()


def _test_ex1():
    # --- empty + all-non-Tensor pass-through ---
    assert unbox_args(()) == ()
    assert unbox_args((1, 2.0, 'x', (3, 4))) == (1, 2.0, 'x', (3, 4))

    # --- single MiniTensor unwrapped ---
    raw1 = t.tensor([1.0, 2.0])
    t1 = MiniTensor(raw1)
    result = unbox_args((t1,))
    assert isinstance(result, tuple), 'must return a tuple, not a list'
    assert len(result) == 1
    assert result[0] is raw1, 'must store the SAME raw tensor (identity, not copy)'

    # --- two MiniTensors at consecutive positions ---
    raw2 = t.tensor([3.0, 4.0])
    t2 = MiniTensor(raw2)
    result = unbox_args((t1, t2))
    assert result == (raw1, raw2)
    assert result[0] is raw1 and result[1] is raw2, 'identity preserved'

    # --- mixed: Tensor then float ---
    result = unbox_args((t1, 3.0))
    assert result == (raw1, 3.0)

    # --- mixed: float then Tensor (order preserved, NOT collapsed) ---
    result = unbox_args((3.0, t1))
    assert result == (3.0, raw1), (
        f'order must be preserved (not collapsed), got {result}'
    )

    # --- mixed batch: int, Tensor, tuple, Tensor ---
    result = unbox_args((5, t1, (1, 2, 3), t2))
    assert result == (5, raw1, (1, 2, 3), raw2)

    # --- raw torch.Tensor MUST pass through (not MiniTensor → unchanged) ---
    raw_passthrough = t.tensor([9.0])
    result = unbox_args((raw_passthrough, t1))
    assert result[0] is raw_passthrough, (
        'raw torch.Tensor must pass through untouched (only MiniTensor gets unboxed)'
    )
    assert result[1] is raw1

    # --- length always matches input length ---
    for inp in [(t1,), (t1, t2), (1, 2, 3, t1, 4), ()]:
        assert len(unbox_args(inp)) == len(inp), (
            f'unbox_args dropped/added entries: input len {len(inp)}, output {unbox_args(inp)}'
        )

    # --- the unboxed value is a torch.Tensor, not a MiniTensor ---
    result = unbox_args((t1,))
    assert isinstance(result[0], t.Tensor), 'unboxed value must be torch.Tensor'
    assert not isinstance(result[0], MiniTensor), (
        'unboxed value must NOT still be a MiniTensor'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def unbox_args(args: tuple) -> tuple:
    return tuple(
        a.array if isinstance(a, MiniTensor) else a
        for a in args
    )
```

**Why `isinstance(a, MiniTensor)` and not `hasattr(a, 'array')`.** Duck-typing on `.array` would catch random objects that happen to expose `.array` — `numpy.ndarray` literally has an `.array` interface protocol. `isinstance` is precise: we want the wrapper class, not anything array-shaped.

**Returns a tuple, not a generator.** A generator would only iterate once — `fwd_fn(*raw_args, **kwargs)` would consume it but Recipe construction (which reuses `raw_args`) would see an exhausted iterator. Tuple is canonical: immutable, re-iterable, cheap.

**Dual of `build_parents`.** Both walk `args` with the same `isinstance` check. `unbox_args` REPLACES each MiniTensor with its `.array`. `build_parents` KEEPS each MiniTensor (filtered, keyed by argnum). Together they're the two outputs of the same scan — sometimes written as one combined helper that returns `(raw_args, parents)` in a single pass.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()